# Prompt Polisher — Full Model Training (Colab T4)

This notebook retrains the **PromptPolisherTransformer** from scratch.

**Architecture:** LLaMA-style decoder (RMSNorm, RoPE, SwiGLU) — 6 layers, 384 hidden, 6 heads, ~35M params

**Pipeline:** Collect corpus → Train tokenizer → Collect SFT data → Train model

**Time:** ~1.5-2 hours total on T4

---
**IMPORTANT:** Set runtime to **GPU → T4** before running.

`Runtime → Change runtime type → T4 GPU`

## Step 0: Setup — Clone repo & install dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/Prompt-Polisher-Org/prompt-polisher.git
%cd prompt-polisher

In [ ]:
# Install exact dependencies
!pip install -q torch>=2.4 sentencepiece>=0.2.0 datasets>=2.19 tqdm>=4.66 tensorboard numpy>=1.26

In [ ]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "ERROR: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## Step 1: Collect tokenizer corpus (~5 min)

Downloads from HuggingFace:
- OpenAssistant/oasst2
- databricks/databricks-dolly-15k
- sahil2801/CodeAlpaca-20k

Output: `ai/data/corpus/tokenizer_corpus.txt`

In [ ]:
!python -m ai.src.tokenizer.collect_corpus

In [ ]:
# Verify corpus was created
import os
corpus_path = "ai/data/corpus/tokenizer_corpus.txt"
size_mb = os.path.getsize(corpus_path) / (1024 * 1024)
with open(corpus_path, 'r', encoding='utf-8') as f:
    line_count = sum(1 for _ in f)
print(f"Corpus: {size_mb:.1f} MB, {line_count:,} lines")

## Step 2: Train SentencePiece BPE tokenizer (~3 min)

Config (must match exactly):
- Algorithm: BPE
- Vocab size: 32,000
- Special tokens: `<pad>`(0), `<unk>`(1), `<bos>`(2), `<eos>`(3), `<sep>`(4)
- Byte fallback: enabled
- Character coverage: 0.9995

Output: `ai/models/tokenizer/prompt_polisher_bpe.model`

In [ ]:
!python -m ai.src.tokenizer.train_tokenizer

In [ ]:
# Verify tokenizer works
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("ai/models/tokenizer/prompt_polisher_bpe.model")

print(f"Vocab size: {sp.get_piece_size()}")
print(f"BOS ID: {sp.bos_id()}")
print(f"EOS ID: {sp.eos_id()}")
print(f"PAD ID: {sp.pad_id()}")
print(f"UNK ID: {sp.unk_id()}")

# Test encode/decode
test = "Optimize the following prompt to be more effective"
ids = sp.encode(test, out_type=int)
decoded = sp.decode(ids)
print(f"\nTest: '{test}'")
print(f"Token IDs: {ids}")
print(f"Decoded: '{decoded}'")
print(f"Fertility: {len(ids)/len(test.split()):.2f} tokens/word")

## Step 3: Collect SFT training data (~5 min)

Downloads and reformats as (bad_prompt -> optimized_prompt) pairs:
- databricks/databricks-dolly-15k
- tatsu-lab/alpaca
- sahil2801/CodeAlpaca-20k

Target: 5,000-10,000 pairs (deduped, shuffled with seed=42)

Output: `ai/data/sft/splits/train.jsonl`, `val.jsonl`, `test.jsonl` (90/5/5 split)

In [ ]:
!python -m ai.src.training.collect_sft_data

In [ ]:
# Verify data splits
import json

for split in ['train', 'val', 'test']:
    path = f"ai/data/sft/splits/{split}.jsonl"
    with open(path, 'r', encoding='utf-8') as f:
        count = sum(1 for _ in f)
    print(f"{split}: {count} examples")

# Show a sample
with open("ai/data/sft/splits/train.jsonl", 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
print(f"\n--- Sample ---")
print(f"Input:  {sample['input'][:100]}...")
print(f"Output: {sample['output'][:100]}...")

## Step 4: Train the model (~1-1.5 hours)

**Exact config matching `best_model.pt`:**

| Parameter | Value |
|-----------|-------|
| Architecture | 6 layers, 384 hidden, 6 heads, 1024 FFN |
| Parameters | ~35.2M |
| Max seq len | 512 |
| Optimizer | AdamW (betas=0.9,0.95, wd=0.01) |
| LR | 3e-4 peak, cosine decay, 500 warmup |
| Batch size | 8 x 4 grad_accum = 32 effective |
| Precision | Mixed FP16 |
| Steps | 11,000 |
| Grad clipping | max_norm 1.0 |

In [ ]:
# Verify model architecture before training
from ai.src.training.config import get_small_config
from ai.src.training.architecture import create_model

config = get_small_config()
model = create_model(config)
param_count = sum(p.numel() for p in model.parameters())
print(config)
print(f"\nExact parameter count: {param_count:,}")
print(f"Expected from best_model.pt: 35,197,824")
assert param_count == 35197824, f"Parameter mismatch! Got {param_count}"
print("Parameter count matches best_model.pt")
del model  # Free memory before training

In [ ]:
# TRAIN - this takes ~1-1.5 hours on T4
# --config small  -> 6L/384D/6H (matches best_model.pt exactly)
# --max-steps 11000  -> same number of steps as the original
!python -m ai.src.training.train --config small --max-steps 11000

In [ ]:
# Verify the trained checkpoint
import torch
import glob

# The training script saves best_model.pt (best val loss) and step_N.pt checkpoints
checkpoints = glob.glob("ai/models/checkpoints/*.pt")
print("Saved checkpoints:")
for cp_path in sorted(checkpoints):
    size_mb = os.path.getsize(cp_path) / (1024 * 1024)
    print(f"  {cp_path} ({size_mb:.1f} MB)")

# Load and inspect best checkpoint
best_path = "ai/models/checkpoints/best_model.pt"
if os.path.exists(best_path):
    cp = torch.load(best_path, map_location='cpu', weights_only=False)
    print(f"\n--- best_model.pt ---")
    print(f"Step: {cp['step']}")
    print(f"Loss: {cp['loss']:.4f}")
    print(f"Config layers: {cp['config']['num_layers']}")
    print(f"Config hidden: {cp['config']['hidden_dim']}")
    print(f"Config heads: {cp['config']['num_heads']}")
    sd = cp['model_state_dict']
    total = sum(v.numel() for v in sd.values())
    print(f"Parameters: {total:,}")
else:
    print("WARNING: best_model.pt not found - using latest step checkpoint")

## Step 5: Quick test — generate a sample prompt

In [ ]:
from ai.src.inference.engine import InferenceEngine

# Load from the best checkpoint
best_path = "ai/models/checkpoints/best_model.pt"
if not os.path.exists(best_path):
    # Fallback to the latest step checkpoint
    checkpoints = sorted(glob.glob("ai/models/checkpoints/step_*.pt"))
    best_path = checkpoints[-1]

engine = InferenceEngine.from_checkpoint(best_path)

# Test prompts
test_prompts = [
    "Write about dogs",
    "Explain quantum computing",
    "Create a Python sorting function",
    "solve 2 raise to x + 3 cubed = 90",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"INPUT:  {prompt}")
    print(f"-"*60)
    result = engine.generate(prompt, temperature=0.7, max_new_tokens=256)
    print(f"OUTPUT: {result['generated_text']}")
    print(f"Tokens: {result['token_count']} | Latency: {result['latency_ms']:.0f}ms")

## Step 6: Download trained files

You need **BOTH** files on your local machine:

1. `prompt_polisher_bpe.model` -> place at `ai/models/tokenizer/`
2. `best_model.pt` -> place at `backend/app/models/` (or `ai/models/checkpoints/`)

In [ ]:
from google.colab import files

# Download the tokenizer (this is the file you were missing)
print("Downloading tokenizer...")
files.download('ai/models/tokenizer/prompt_polisher_bpe.model')

# Download the trained model
print("Downloading best_model.pt...")
best_path = "ai/models/checkpoints/best_model.pt"
if not os.path.exists(best_path):
    checkpoints = sorted(glob.glob("ai/models/checkpoints/step_*.pt"))
    best_path = checkpoints[-1]
files.download(best_path)

## (Optional) Save to Google Drive

So you don't lose the files if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/prompt-polisher-model'
os.makedirs(save_dir, exist_ok=True)

# Copy tokenizer
shutil.copy('ai/models/tokenizer/prompt_polisher_bpe.model', save_dir)
print(f"Tokenizer saved to {save_dir}")

# Copy best model
shutil.copy(best_path, save_dir)
print(f"Model saved to {save_dir}")

# Copy tokenizer config
if os.path.exists('ai/models/tokenizer/tokenizer_config.json'):
    shutil.copy('ai/models/tokenizer/tokenizer_config.json', save_dir)

print(f"\nAll files saved to Google Drive: {save_dir}")
!ls -la {save_dir}

---

## After downloading — Local setup

Place the files:
```
prompt-polisher/
  ai/models/tokenizer/prompt_polisher_bpe.model   <- tokenizer
  backend/app/models/best_model.pt                 <- model
```

Then start the inference server:
```bash
python -m uvicorn ai.src.inference.server:app --host 0.0.0.0 --port 8001
```

It should now say **Model loaded successfully** instead of rule-based mode.